In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")

In [ ]:
list_unreachable_tokens = []
for i in range(256, 128000): 
    decoded_id = tokenizer.encode(tokenizer.decode(i))[1:] 
    if len(decoded_id) > 1:
        list_unreachable_tokens.append(i)
        continue
    
    if decoded_id[0] != i:
        list_unreachable_tokens.append(i)
        continue

In [ ]:
list_undertrained = []
with open('./meta_llama_Meta_Llama_3_1_8B.md', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        splits = line.split('|')
        if len(splits) == 7:
            try:
                list_undertrained.append(int(splits[1].strip()))
            except:
                continue

In [ ]:
len(list_undertrained), len(list_unreachable_tokens)

In [ ]:
final_list_to_replace = list_undertrained
for token_id in list_unreachable_tokens:
    if token_id not in final_list_to_replace:
        final_list_to_replace.append(token_id)
len(final_list_to_replace)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", use_fast=False)
tokenizer.save_pretrained("./Llama-3.1-8B-Base/")

import json
tokenizer_json = json.load(open("./Llama-3.1-8B-Base/tokenizer.json",encoding='utf-8'))
merge_list = tokenizer_json['model']['merges']


import networkx as nx
G = nx.DiGraph()
for item in merge_list:
    item_token = ''.join(item)
    G.add_edge(item[0],item_token)
    G.add_edge(item[1],item_token)


In [ ]:
final_tokens_to_replace = [tokenizer.convert_ids_to_tokens(token_id) for token_id in final_list_to_replace]


def is_accessible_nodes(graph, start_node):
    if len(list(nx.descendants(graph, start_node))) > 0 :
        return True
    
    return False

final_tokens_can_replace = []
for node in final_tokens_to_replace:
    try:
        if not is_accessible_nodes(G,node):
            final_tokens_can_replace.append(tokenizer.convert_tokens_to_ids(node))
            print(node)
    except: pass

with open('./TokenIDs_Can_Replace_Llama3.1-Magikarp+Unreachable.txt','w',encoding='utf-8') as f:
    for token_id in final_tokens_can_replace:
        f.write(str(token_id) + '\n')
f.close()


In [ ]:
import json
tokenizer_json = json.load(open("./Llama-3.1-8B-Base/tokenizer.json",encoding='utf-8'))

In [ ]:
tokenids_to_replace = open('./TokenIDs_Can_Replace_Llama3.1-Magikarp+Unreachable.txt','r',encoding='utf-8').read().splitlines()
tokenids_to_replace = [int(x.strip()) for x in tokenids_to_replace]

In [ ]:
from transformers import AutoTokenizer
base_tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B')
tokens_to_replace = [base_tokenizer.convert_ids_to_tokens(x) for x in tokenids_to_replace]
tokens_to_replace

In [ ]:
len(tokens_to_replace)

In [ ]:
tokenizer_json_dump = {}

for key,val in tokenizer_json.items():
    if key == 'model': continue
    else: tokenizer_json_dump[key] = val

In [ ]:
tokenizer_json_dump['model'] = {}
for key,val in tokenizer_json['model'].items():
    if key == 'vocab' or key == 'merges': continue
    else: tokenizer_json_dump['model'][key] = val

In [ ]:
tokens_to_add = []
for line in open('./VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Llama3.1_Vocab/10.0_.txt','r',encoding='utf-8'):
    tokens_to_add.append(line.strip())

tokens_to_add = sorted(tokens_to_add, key = lambda x: len(x))
len(tokens_to_add), len([x for x in tokens_to_add if len(base_tokenizer.tokenize(x.replace('Ġ',' '))) > 2])

In [ ]:
tokens_dump = {}
for k in tokens_to_add:
    # if 'Ĕ' in k: continue
    # if 'Ä' in k: continue
    # if 'ą' in k: continue
        
    split = base_tokenizer.tokenize(k if not k.startswith('Ġ') else ' '+k[1:])
    
    # flag = False
    # for sp in split:
    #     if 'Ä' in sp: 
    #         flag = True
    #         break
        
    # if flag: continue

    
    if len(split) == 1:
        continue
    
    elif len(split) == 2:
        tokens_dump[k] = [split[0],split[1]]
    
    if len(split) > 2:
        new_word = split[0]
        for i in range(1,len(split)):
            left = new_word
            right = split[i]
            new_word += split[i]
            if new_word not in tokens_dump:
                tokens_dump[new_word] = [left,right]
            

len(tokens_dump)        


In [ ]:
final_tokens_to_add = []
for key,val in tokens_dump.items():
    if len(final_tokens_to_add) == len(tokenids_to_replace): break
    
    final_tokens_to_add.append([key]+val)

len(final_tokens_to_add)

In [ ]:
final_tokens_to_add

In [ ]:
len(final_tokens_to_add), len(tokenids_to_replace)

In [ ]:
tokenizer_json_dump['model']['vocab'] = {}

tokens_replaced = []

for key,val in tokenizer_json['model']['vocab'].items():
        tokenizer_json_dump['model']['vocab'][key] = val

idx = 0
for key,val in zip(tokens_to_replace,tokenids_to_replace):
    if idx == len(final_tokens_to_add): break
    
    print('Replacing:',key,val,'With:',final_tokens_to_add[idx][0])
    
    tokenizer_json_dump['model']['vocab'][final_tokens_to_add[idx][0]] = val
    del tokenizer_json_dump['model']['vocab'][key]
    tokens_replaced.append([val,key,final_tokens_to_add[idx][0],base_tokenizer.encode(final_tokens_to_add[idx][0].replace('Ġ',' '))[1:]])
    idx += 1

In [ ]:
final_tokens_to_add

In [ ]:
len(final_tokens_to_add), len(tokens_dump)

In [ ]:
count = 0
tokenizer_json_dump['model']['merges'] = []
tokenizer_json_dump['model']['merges'] = [tokens_dump[x[0]] for x in final_tokens_to_add]
for item in tokenizer_json['model']['merges']:
    if ''.join(item) in [x[1] for x in tokens_replaced]:
        print('Skipping:',item)
        count += 1
        continue
    else:
        tokenizer_json_dump['model']['merges'].append(item)
        
print(count)

In [ ]:
for key in tokenizer_json_dump['model']['vocab']:
    print(key)

In [ ]:
len(tokenizer_json_dump['model']['vocab'])

In [ ]:
for key,val in tokens_dump.items():
    if key not in tokenizer_json_dump["model"]["vocab"]:
        print(key,len(tokenizer_json_dump["model"]["vocab"]))
        tokenizer_json_dump["model"]["vocab"][key] = len(tokenizer_json_dump["model"]["vocab"])
        tokenizer_json_dump["model"]["merges"].append(val)


In [ ]:
len(tokenizer_json_dump["model"]["vocab"]), len(tokenizer_json_dump["model"]["merges"])

In [ ]:
dir_tokenizer = './VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Llama3.1_Tokenizers/SupremeCourtCase_DomainSpecific_MEDVOC_10.0_Replaced_MagikarpUnreachable/'
base_tokenizer.save_pretrained(dir_tokenizer)

with open(f'{dir_tokenizer}/tokenizer.json','w',encoding='utf-8') as f:
    json.dump(tokenizer_json_dump,f)
f.close()

In [ ]:
import pickle as pkl
with open(f'{dir_tokenizer}/replaced_tokens.pkl','wb') as f:
    pkl.dump(tokens_replaced,f)
f.close()

In [1]:
from transformers import AutoTokenizer
dir_tokenizer = './VocabFiles/CPT-SupremeCourtCase/CPT-SupremeCourtCase-Llama3.1_Tokenizers/SupremeCourtCase_DomainSpecific_MEDVOC_10.0_Replaced_MagikarpUnreachable/'

modi_tokenizer = AutoTokenizer.from_pretrained(dir_tokenizer)

In [ ]:
modi_tokenizer.save_pretrained(dir_tokenizer)